# Why NLP, Why Now — companion notebook

Chapter 1 of *Applied Natural Language Processing — From Tokens to Agents*.
Read the chapter first: **[Why NLP, Why Now](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/01-why-nlp-now.html)**.

Chapter 1 argues rather than measures. Each of its intuition checks points at a later
chapter that settles the question with a number. This notebook lets you watch three of
the chapter's central claims happen on real data, on your own machine:

1. **A percentage means nothing without its baseline** (intuition check 1).
2. **Three ways to solve the same problem** trade flexibility against cost, latency and
   predictability (the chapter's comparison table).
3. **Agreement is not reliability** (intuition check 3).

**How to read the numbers.** Everything printed below is what *this run* produced. Where
the book reports the same quantity, the cell prints the book's value beside it and says
whether the two match. Timings depend on your machine and will not match anyone's; read
them as orders of magnitude and as ratios.

**Runtime.** Under a minute on a laptop CPU after the first download (about 25 MB).
Section 3c adds under a minute if a local `ollama` server is running, and skips itself
cleanly if one is not.

## 1. Setup

The first cell records the interpreter and package versions this run used. If your
numbers differ from the ones saved in this notebook, compare this cell first.

In [1]:
import hashlib
import os
import platform
import sys
import time
from collections import Counter
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split

SEED = 42

print(f"Python        {platform.python_version()}  ({platform.system()} {platform.machine()})")
for pkg in ["numpy", "pandas", "scikit-learn", "pyarrow", "requests"]:
    print(f"{pkg:<13} {version(pkg)}")

Python        3.12.13  (Darwin arm64)
numpy         2.5.2
pandas        3.0.5
scikit-learn  1.9.0
pyarrow       25.0.1
requests      2.34.2


### The corpus

The data is the one [chapter 4](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html) uses: **Turkish product reviews labelled
positive or negative**, 235,165 rows scraped from a Turkish e-commerce site. Short,
informal and unedited, which is the point.

**Licence.** The dataset was compiled by Fatih Barmanbay and is distributed on Hugging
Face as `fthbrmnby/turkish_product_reviews` under **CC BY-SA 4.0**. That is share-alike,
a stricter licence than the book's own CC BY 4.0, so check it before you redistribute
anything derived from it. Chapter 4 also notes what the dataset card does *not* say:
which site was scraped, when, and how the positive/negative label was derived.

The loading code below is chapter 4's, line for line: rename the columns, strip
whitespace, drop exact duplicate texts. The only addition is a local cache, so that
re-running the notebook does not download the file again.

**What to look at:** the `this run` and `book` columns. Chapter 4 printed 235,165 rows,
233,508 after de-duplication, and 93.7% positive.

In [2]:
URL = ("https://huggingface.co/datasets/fthbrmnby/turkish_product_reviews/"
       "resolve/main/data/train-00000-of-00001.parquet")
CACHE = Path("data/cache/turkish_product_reviews.parquet")

if not CACHE.exists():
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    CACHE.write_bytes(response.content)
print(f"parquet file  {CACHE.stat().st_size:,} bytes, "
      f"sha256 {hashlib.sha256(CACHE.read_bytes()).hexdigest()[:12]}")


def vs_book(label, ours, book, fmt="{:,}"):
    """Print this run's value beside the book's, compared at the book's printed precision."""
    ours_s, book_s = fmt.format(ours), fmt.format(book)
    verdict = "same as the book" if ours_s == book_s else "DIFFERS from the book"
    print(f"{label:<30} this run {ours_s:>9}   book {book_s:>9}   {verdict}")


# chapter 4's loading code
raw = pd.read_parquet(CACHE)
df = raw.rename(columns={"sentence": "text", "sentiment": "label"})
df["text"] = df["text"].astype(str).str.strip()
df = df.drop_duplicates(subset="text")

print()
vs_book("rows as downloaded", len(raw), 235_165)
vs_book("after de-duplication", len(df), 233_508)
vs_book("duplicates removed", len(raw) - len(df), 1_657)
vs_book("label 1 (positive)", int((df["label"] == 1).sum()), 218_747)
vs_book("label 0 (negative)", int((df["label"] == 0).sum()), 14_761)
vs_book("positive share, %", 100 * (df["label"] == 1).mean(), 93.7, "{:.1f}")

parquet file  24,354,762 bytes, sha256 a5ea59cd4f78

rows as downloaded             this run   235,165   book   235,165   same as the book
after de-duplication           this run   233,508   book   233,508   same as the book
duplicates removed             this run     1,657   book     1,657   same as the book
label 1 (positive)             this run   218,747   book   218,747   same as the book
label 0 (negative)             this run    14,761   book    14,761   same as the book
positive share, %              this run      93.7   book      93.7   same as the book


## 2. A percentage without its baseline

Intuition check 1 asks: *a vendor's model is 94% accurate on your task — is that good?*
The chapter's answer is that you cannot tell, because the missing number is the base
rate. The habit it asks you to form is to **ask what the majority class rate is, and
subtract**.

First the split. Chapter 4 describes it as a stratified 50,000-review subsample, split
80/20 with seed 42. Chapter 4 itself warns that this description is under-specified: it
names a family of procedures that disagree. [Chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html) recovered the exact
call for the 50,000, a single stratified `train_test_split`, and the cell uses that call.
The 80/20 step is our reading of chapter 4's prose, not code the book prints, so the cell
checks the result against the book's counts instead of assuming it.

Chapter 12's advice is to print a fingerprint of the sample alongside the call, so the
cell does that too. The sharpest fingerprint, the feature count of chapter 4's
vectoriser, arrives in section 3b.

In [3]:
# chapter 12's recovered call for the 50,000-review subsample
sub, _ = train_test_split(df, train_size=50_000, stratify=df["label"], random_state=SEED)
# the 80/20 split: our reading of chapter 4's "stratified, seed 42"
train, test = train_test_split(sub, test_size=0.2, stratify=sub["label"], random_state=SEED)

print("subsample fingerprint", hashlib.sha256(",".join(map(str, sorted(sub.index))).encode()).hexdigest()[:12])
print()
vs_book("train size", len(train), 40_000)
vs_book("train positive", int((train["label"] == 1).sum()), 37_471)
vs_book("train negative", int((train["label"] == 0).sum()), 2_529)
vs_book("test size", len(test), 10_000)
vs_book("test positive", int((test["label"] == 1).sum()), 9_368)
vs_book("test negative", int((test["label"] == 0).sum()), 632)

y_test = test["label"].to_numpy()
y_train = train["label"].to_numpy()

subsample fingerprint 1e724ccf17fa

train size                     this run    40,000   book    40,000   same as the book
train positive                 this run    37,471   book    37,471   same as the book
train negative                 this run     2,529   book     2,529   same as the book
test size                      this run    10,000   book    10,000   same as the book
test positive                  this run     9,368   book     9,368   same as the book
test negative                  this run       632   book       632   same as the book


### The laziest possible model

Predict "positive" for every review and never look at the text. The helper `score`
defined here is used for every approach in this notebook, so all of them are measured the
same way:

- **accuracy** is the share of reviews labelled correctly;
- **F1(neg)** is the F1 score on the negative class, the complaints;
- **macro-F1** is the unweighted mean of the two per-class F1 scores, so the rare class
  counts as much as the common one.

**What to look at:** the gap between accuracy and macro-F1, and the confusion matrix.

In [4]:
def score(y_true, y_pred):
    """Accuracy, macro-F1 and negative-class F1, in percent."""
    f1_neg, f1_pos = f1_score(y_true, y_pred, labels=[0, 1], average=None, zero_division=0)
    return {"accuracy": 100 * accuracy_score(y_true, y_pred),
            "macro-F1": 100 * (f1_neg + f1_pos) / 2,
            "F1(neg)": 100 * f1_neg}


def show_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(pd.DataFrame(cm, index=["actual neg", "actual pos"],
                       columns=["predicted neg", "predicted pos"]).to_string())


majority_pred = np.ones_like(y_test)
majority = score(y_test, majority_pred)

show_confusion(y_test, majority_pred)
print()
vs_book("accuracy", majority["accuracy"], 93.68, "{:.2f}")
vs_book("macro-F1", majority["macro-F1"], 48.37, "{:.2f}")
vs_book("F1(neg)", majority["F1(neg)"], 0.00, "{:.2f}")

            predicted neg  predicted pos
actual neg              0            632
actual pos              0           9368

accuracy                       this run     93.68   book     93.68   same as the book
macro-F1                       this run     48.37   book     48.37   same as the book
F1(neg)                        this run      0.00   book      0.00   same as the book


**Reading it.** The model has no inputs and no parameters, and its accuracy is simply the
share of positive reviews in the test set. A vendor quoting "94% accurate" on data like
this is quoting a number less than half a point above a constant's.

Macro-F1 exposes it: the negative class gets an F1 of zero, and the average of zero and
the positive class's F1 lands below 50. That is why [chapter 4](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html) makes
macro-F1 the headline and reports accuracy only to show how misleading it is, and why
[chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html#sec-classification-metrics) returns to metrics in depth.

Keep both numbers in view. Every approach below has to be read against them.

## 3. Three ways to solve the same problem

The chapter's table compares three approaches: counting words and fitting a classifier,
fine-tuning a pretrained encoder, and prompting a large language model. Its summary is
that **the newer approaches buy flexibility and pay for it in cost, latency, and
predictability**.

This section runs three approaches that fit on a laptop CPU in minutes, on the same test
reviews, scored the same way:

- **3a. A hand-written rule.** Not a column of the table: it is the first era on the
  chapter's timeline, when people *legislated* language instead of estimating it. It is
  here because it is what most people try first.
- **3b. The classical column:** TF-IDF features and logistic regression, exactly as
  chapter 4 builds it.
- **3c. The LLM column:** prompting a small open-weight model through a local `ollama`
  server.

The fine-tuned encoder column is not run here, because training one is a job for a GPU.
[Chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html) fine-tunes one on this corpus and measures it at about 2 ms per
document, batched, which is the figure the chapter's table quotes.

### 3a. A rule you write by hand

Twelve negative phrases and ten positive ones, written once and not tuned on the test
set. The decision has the same shape as the crude annotator in
[chapter 5](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html): call a review negative only when a negative phrase appears and
no positive phrase does; otherwise fall back to the majority class.

It is **not** chapter 5's lexicon. That one has 36 positive and 37 negative phrases and
the book does not print them, so do not expect this rule's numbers to match chapter 5's.

Two preprocessing steps come from chapter 4. `tr_lower` lowercases the Turkish way,
because Python's `str.lower()` maps `I` to `i` where Turkish needs `ı`. And the rule
deletes U+0307 (COMBINING DOT ABOVE), a stray mark chapter 4 found in about 8.4% of its
working subsample; without that step, a review containing `deği̇l` would not match the
phrase `değil`.

Matching is by substring, so `iade` ("refund") also matches `iadesi`. That is a feature
and a bug at the same time, and exercise 2 asks you to find the bug.

In [5]:
TR_LOWER = str.maketrans({"I": "ı", "İ": "i"})


def tr_lower(s):
    return s.translate(TR_LOWER).lower()


NEG = ["kötü", "berbat", "rezalet", "değil", "beğenmedim", "memnun kalmadım",
       "tavsiye etmem", "tavsiye etmiyorum", "iade", "bozuk", "kalitesiz", "pişman"]
POS = ["güzel", "harika", "mükemmel", "süper", "memnunum", "teşekkür",
       "tavsiye ederim", "beğendim", "kaliteli", "başarılı"]


def rule_label(text):
    t = tr_lower(text).replace("\u0307", "")
    neg = any(p in t for p in NEG)
    pos = any(p in t for p in POS)
    return 0 if (neg and not pos) else 1


start = time.perf_counter()
rule_pred = np.array([rule_label(t) for t in test["text"]])
rule_us_per_doc = 1e6 * (time.perf_counter() - start) / len(test)
rule = score(y_test, rule_pred)

show_confusion(y_test, rule_pred)
print()
for k, v in rule.items():
    print(f"{k:<9} {v:6.2f}   (majority baseline {majority[k]:6.2f})")
print(f"latency   {rule_us_per_doc:.1f} µs per review, pure Python, one at a time")

            predicted neg  predicted pos
actual neg            200            432
actual pos            303           9065

accuracy   92.65   (majority baseline  93.68)
macro-F1   65.67   (majority baseline  48.37)
F1(neg)    35.24   (majority baseline   0.00)
latency   8.6 µs per review, pure Python, one at a time


**What to look at:** accuracy *below* the constant model's, macro-F1 well above it. The
rule finds some complaints, and pays for them with false alarms on positive reviews that
happen to contain a negative word.

The chapter explains why rules stall: *language has an enormous tail*. Here are a few
reviews labelled negative that the rule called positive. Try to write the phrase that
would catch each one without breaking something else. Read them carefully first: not
every miss is the rule's fault, and chapter 12 finds that on this corpus the most
confident mistakes are often label errors.

In [6]:
missed = test[(y_test == 0) & (rule_pred == 1)]
print(f"{len(missed)} of {int((y_test == 0).sum())} negative test reviews were labelled positive. Five of them:\n")
for text in missed.sample(5, random_state=SEED)["text"]:
    print("-", text[:160] + ("…" if len(text) > 160 else ""))

432 of 632 negative test reviews were labelled positive. Five of them:

- artıları:1-fiyatı uygun
- bazi arkadaslar yazmis kucuk falan diyede tam kestiremedim aldim ama baya baya kucuk.biraz daha buyuk olsaydi iyiydi ama bu boyutuyla cok kotu.zincir baya cansi…
- ürün güzel ancak duvara çok yakın bu nedenle tvnin arkasına takmanız gereken kablo bağlantılarını takmakta zorlanabilirsiniz. ayrıca tvyi askı aparatına asarken…
- ben pembesini kullanıyorum su kaynatmaya başlarken çok sesli kullandıkca ses artttı 3 4ay içinde rengi lekelenip beyazlaşmasya başladı uzun süre kullanılmayacağ…
- güneş altında kaldığından yaklaşık 2 sene içinde çürüdü plastiği kırılmaya başladı. gölge altında kullanılması belki ömrünü uzatır. ancak ipi sağlam ve güvenli …


### 3b. Counting words and fitting a classifier

This is chapter 4's best representation: TF-IDF over word unigrams and bigrams, fed to a
class-weighted logistic regression. The settings are chapter 4's, including
`lowercase=False` with `preprocessor=tr_lower`, which stops scikit-learn from calling
Python's `str.lower()`.

This cell is also the sharpest reproduction check in the notebook. Chapter 4 warns that
before comparing anything to its numbers you should match its split sizes *and* the
**101,659 features** this vectoriser produces; chapter 12 explains why the feature count
is the convincing one: two different 50,000-review samples have no reason to agree on a
six-digit count. Chapter 4 printed 92.57 accuracy, 76.21 macro-F1 and 56.47 F1(neg).

**What to look at:** the feature count, the three scores, and the training time.
Chapter 1 cites a sparse model on this corpus that trains in about a second
(chapter 4 timed the fit at 1.0 s and, separately, 0.9 s).

In [7]:
PAT = r"[a-zçğıöşü]+"
vec = TfidfVectorizer(min_df=2, sublinear_tf=True, ngram_range=(1, 2), token_pattern=PAT,
                      preprocessor=tr_lower, lowercase=False)
clf = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)

start = time.perf_counter()
X_train = vec.fit_transform(train["text"])
vectorise_s = time.perf_counter() - start
clf.fit(X_train, y_train)
train_s = time.perf_counter() - start

tfidf_pred = clf.predict(vec.transform(test["text"]))
tfidf = score(y_test, tfidf_pred)

show_confusion(y_test, tfidf_pred)
print()
vs_book("features", X_train.shape[1], 101_659)
vs_book("accuracy", tfidf["accuracy"], 92.57, "{:.2f}")
vs_book("macro-F1", tfidf["macro-F1"], 76.21, "{:.2f}")
vs_book("F1(neg)", tfidf["F1(neg)"], 56.47, "{:.2f}")
print(f"\ntraining time {train_s:.2f} s on 40,000 reviews "
      f"({vectorise_s:.2f} s building features, {train_s - vectorise_s:.2f} s fitting)")

            predicted neg  predicted pos
actual neg            482            150
actual pos            593           8775

features                       this run   101,659   book   101,659   same as the book
accuracy                       this run     92.57   book     92.57   same as the book
macro-F1                       this run     76.21   book     76.21   same as the book
F1(neg)                        this run     56.47   book     56.47   same as the book

training time 0.97 s on 40,000 reviews (0.88 s building features, 0.09 s fitting)


**Reading it.** Accuracy is again below the constant model's, and macro-F1 is far above
both the constant and the hand-written rule. Nobody wrote a phrase list here; the model
estimated one weight per word and word pair from 40,000 labelled reviews.

The chapter's table claims a classical model **explains its decision, exactly: a linear
score decomposes into per-term contributions**. The next cell shows what that means. It
prints the largest weights in each direction, then takes the first negative test review
the model is confident about (a score below −2, with six to twelve active terms) and adds
up its terms. Positive weights push towards "positive"; the intercept plus the
contributions *is* the model's score, and the cell checks that.

In [8]:
terms = vec.get_feature_names_out()
weights = clf.coef_[0]                       # classes_ are [0, 1]: positive weight -> label 1
order = np.argsort(weights, kind="stable")
print("towards NEGATIVE                  towards POSITIVE")
for lo, hi in zip(order[:10], order[::-1][:10]):
    print(f"{terms[lo]:<22} {weights[lo]:+.2f}     {terms[hi]:<22} {weights[hi]:+.2f}")

# the first negative review the model is confident about, with a handful of active terms
X_test = vec.transform(test["text"])
nnz = np.diff(X_test.indptr)
test_scores = clf.decision_function(X_test)
i = next(j for j in range(len(test)) if y_test[j] == 0 and test_scores[j] < -2 and 6 <= nnz[j] <= 12)
row = X_test[i]
contrib = sorted(zip(terms[row.indices], row.data * weights[row.indices]), key=lambda t: t[1])

print(f"\nreview: {test['text'].iloc[i]}\n")
for term, c in contrib:
    print(f"  {term:<28} {c:+.3f}")
total = clf.intercept_[0] + sum(c for _, c in contrib)
print(f"  {'(intercept)':<28} {clf.intercept_[0]:+.3f}")
print(f"  {'score = sum of the above':<28} {total:+.3f}   "
      f"model's own decision_function {clf.decision_function(row)[0]:+.3f}  (< 0 means negative)")

towards NEGATIVE                  towards POSITIVE
değil                  -7.90     gayet                  +8.07
kötü                   -6.80     teşekkürler            +7.21
fakat                  -6.05     güzel                  +6.44
beğenmedim             -5.93     tavsiye ederim         +6.32
tavsiye etmiyorum      -5.83     ederim                 +6.30
etmiyorum              -5.58     mükemmel               +6.17
kalitesiz              -5.37     harika                 +6.06
iade                   -5.36     gerçekten              +5.69
ancak                  -4.94     memnunum               +5.47
ama                    -4.75     süper                  +4.99



review: çekiş gücü aşırı derecede düşük aranan performans yok

  düşük                        -0.950
  aşırı                        -0.706
  aşırı derecede               -0.369
  çekiş gücü                   -0.307
  gücü                         -0.274
  derecede                     -0.254
  çekiş                        -0.232
  yok                          -0.215
  aranan                       +0.115
  performans                   +0.510
  (intercept)                  +0.262
  score = sum of the above     -2.420   model's own decision_function -2.420  (< 0 means negative)


Now the table's latency row, which says a classical model runs in **microseconds per
document, batched**. The cell times prediction on all 10,000 test reviews at once (the
median of five runs), and then one review at a time over 500 reviews.

For comparison, [chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html) measured this pipeline at 0.014 s per 1,000
documents batched and 0.153 s per 1,000 one at a time, that is 14 µs and 153 µs per
document, on a different machine.

In [9]:
batched = []
for _ in range(5):
    start = time.perf_counter()
    clf.predict(vec.transform(test["text"]))
    batched.append(time.perf_counter() - start)
tfidf_us_batched = 1e6 * np.median(batched) / len(test)

single = []
for text in test["text"].iloc[:500]:
    start = time.perf_counter()
    clf.predict(vec.transform([text]))
    single.append(time.perf_counter() - start)
tfidf_us_single = 1e6 * np.median(single)

print(f"batched       {tfidf_us_batched:7.1f} µs per review  (10,000 reviews, median of 5 runs)")
print(f"one at a time {tfidf_us_single:7.1f} µs per review  (median over 500 reviews)")

summary = pd.DataFrame(
    [["majority constant", *majority.values(), "none", "—"],
     ["hand-written rule", *rule.values(), "none (hand-written)", f"{rule_us_per_doc:.1f} µs"],
     ["tf-idf + logistic regression", *tfidf.values(), f"{train_s:.2f} s, 40,000 labels", f"{tfidf_us_batched:.1f} µs"]],
    columns=["approach", "accuracy", "macro-F1", "F1(neg)", "training", "latency per review"],
).set_index("approach")
print()
print(summary.round(2).to_string())

batched          16.8 µs per review  (10,000 reviews, median of 5 runs)
one at a time   146.6 µs per review  (median over 500 reviews)

                              accuracy  macro-F1  F1(neg)               training latency per review
approach                                                                                           
majority constant                93.68     48.37     0.00                   none                  —
hand-written rule                92.65     65.67    35.24    none (hand-written)             8.6 µs
tf-idf + logistic regression     92.57     76.21    56.47  0.97 s, 40,000 labels            16.8 µs


### 3c. Prompting a local LLM

The last column of the table: no training labels at all, just an instruction. The model
is `llama3.2:3b`, a 3-billion-parameter open-weight model served by a local `ollama`
server.

**The guards.** [Chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html#sec-classification-llm) describes a run that
scored 93.75% accuracy while being completely broken: a reasoning model spent its whole
token budget thinking, every answer came back empty, and every empty answer fell through
to the majority class. The harness below is built so that cannot happen quietly:

- `temperature` 0 and a fixed `seed`, with one fixed prompt;
- `think: false`, which chapter 12 found changed nothing on this model and fixed the
  broken reasoning-model run; it is here in case you swap the model;
- `num_predict` is only a cap, so every call records `done_reason` and the length of any
  `thinking` text;
- **an unparseable answer is not a prediction.** It is counted as a third outcome and
  never mapped to a class. Scores are reported over the answers that parsed, beside the
  parse rate;
- the run aborts if the first 20 answers parse below 50%;
- one warm-up call loads the model and is excluded from timing, and the latency reported
  is the median, not the mean.

**The sample.** 100 test reviews, 50 negative and 50 positive, drawn with seed 42. A
stratified 100 would hold only about six complaints, too few for macro-F1 to mean
anything. The balance changes the baseline: on this sample the constant model scores
50% accuracy, not 93.68%, so read every number in this subsection against 50. Following
chapter 12's rule, the rule and TF-IDF models are re-scored on the same 100 reviews.

**The prompt** is written in English. Chapter 12 found that an English instruction was
worth +16.90 macro-F1 on this model compared with a Turkish one. The book does not print
its prompt, so the one below is new and this is a new measurement, not a reproduction.

The cell skips itself if the server does not answer, if the model is not pulled, or if
the environment variable `ANLP_SKIP_LLM=1` is set.

In [10]:
OLLAMA_URL = os.environ.get("ANLP_OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = "llama3.2:3b"
PROMPT = ("Classify the sentiment of this Turkish product review.\n"
          "Answer with exactly one word: positive or negative.\n\n"
          "Review: {review}\n\nAnswer:")

llm_ready = False
if os.environ.get("ANLP_SKIP_LLM") == "1":
    print("SKIPPED: ANLP_SKIP_LLM=1 is set. Section 3c will not run.")
else:
    try:
        server_version = requests.get(f"{OLLAMA_URL}/api/version", timeout=3).json()["version"]
        models = {m["name"]: m for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=3).json()["models"]}
        if MODEL in models:
            llm_ready = True
            m = models[MODEL]
            print(f"ollama {server_version} at {OLLAMA_URL}")
            print(f"model  {MODEL}  digest {m['digest'][:12]}  "
                  f"{m['details'].get('parameter_size', '?')} parameters, "
                  f"{m['details'].get('quantization_level', '?')}, {m['size'] / 1e9:.1f} GB")
        else:
            print(f"SKIPPED: ollama {server_version} is running but {MODEL} is not pulled "
                  f"(run `ollama pull {MODEL}`). Section 3c will not run.")
    except requests.RequestException as err:
        print(f"SKIPPED: no ollama server answered at {OLLAMA_URL} ({type(err).__name__}). "
              "Section 3c will not run; everything else in the notebook still does.")

ollama 0.24.0 at http://127.0.0.1:11434
model  llama3.2:3b  digest a80c4f17acd5  3.2B parameters, Q4_K_M, 2.0 GB


In [11]:
def ask(review):
    """One call to the model. Returns the raw record; parsing happens separately."""
    payload = {"model": MODEL, "prompt": PROMPT.format(review=review), "stream": False,
               "think": False, "options": {"temperature": 0, "seed": SEED, "num_predict": 8}}
    start = time.perf_counter()
    r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=120)
    if r.status_code == 400 and "think" in r.text:     # a model that rejects the field
        payload.pop("think")
        r = requests.post(f"{OLLAMA_URL}/api/generate", json=payload, timeout=120)
    secs = time.perf_counter() - start
    r.raise_for_status()
    body = r.json()
    return {"raw": body.get("response", ""), "done_reason": body.get("done_reason"),
            "eval_count": body.get("eval_count"), "think_chars": len(body.get("thinking") or ""),
            "secs": secs}


def parse(raw):
    """'positive' -> 1, 'negative' -> 0, anything else -> None (unparseable, never a class)."""
    word = raw.strip().strip(".!*\"'` \n").lower()
    return {"positive": 1, "negative": 0}.get(word)


llm_records = None
if llm_ready:
    sample = pd.concat([test[test["label"] == 0].sample(50, random_state=SEED),
                        test[test["label"] == 1].sample(50, random_state=SEED)])
    sample = sample.sample(frac=1, random_state=SEED)   # interleave the classes over time

    ask("Ürün güzel.")                                   # warm-up: loads the model, not timed
    records = []
    for n, (idx, row) in enumerate(sample.iterrows(), start=1):
        rec = ask(row["text"])
        rec.update(index=idx, y=int(row["label"]), pred=parse(rec["raw"]))
        records.append(rec)
        if n == 20:
            rate = sum(r["pred"] is not None for r in records) / n
            if rate < 0.5:
                print(f"ABORTED after {n} answers: parse rate {rate:.0%}. "
                      "Inspect the raw answers below before trusting any score.")
                for r in records[:5]:
                    print(repr(r["raw"]), r["done_reason"], r["think_chars"])
                break
    else:
        llm_records = pd.DataFrame(records).set_index("index")

    if llm_records is not None:
        parsed = llm_records["pred"].notna()
        print(f"answers        {len(llm_records)}")
        print(f"parsed         {int(parsed.sum())}")
        print(f"unparseable    {int((~parsed).sum())}")
        print(f"done_reason    {dict(sorted(Counter(llm_records['done_reason']).items()))}")
        print(f"thinking chars {int(llm_records['think_chars'].sum())} in total")
        print(f"raw answers    {dict(Counter(llm_records['raw']).most_common())}")
        print(f"latency        median {llm_records['secs'].median():.3f} s per review "
              f"(range {llm_records['secs'].min():.3f}–{llm_records['secs'].max():.3f} s)")
        if (~parsed).any():
            print("\nunparseable answers:", llm_records.loc[~parsed, "raw"].tolist())
else:
    print("SKIPPED: the model was not called (see the cell above).")

answers        100
parsed         100
unparseable    0
done_reason    {'stop': 100}
thinking chars 0 in total
raw answers    {'positive.': 28, 'Negative.': 26, 'positive': 22, 'Negative': 15, 'negative': 5, 'negative.': 3, 'Positive.': 1}
latency        median 0.105 s per review (range 0.093–0.172 s)


**Scoring on the same reviews.** The next cell puts all four approaches on the reviews the
LLM answered: with no unparseable answers, all 100. The interval beside macro-F1 is a 95%
percentile bootstrap over those reviews (2,000 resamples). It is there because 100
reviews cannot resolve small differences, and the width of the interval is the honest way
to say so.

In [12]:
def bootstrap_macro_f1(y_true, y_pred, n_boot=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    stats = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        stats.append(score(y_true[idx], y_pred[idx])["macro-F1"])
    return np.percentile(stats, [2.5, 97.5])


if llm_records is None:
    print("SKIPPED: no LLM answers to score (see the cells above).")
else:
    answered = llm_records[llm_records["pred"].notna()]
    y_s = answered["y"].to_numpy()
    positions = test.index.get_indexer(answered.index)
    preds = {
        "majority constant": np.ones_like(y_s),
        "hand-written rule": rule_pred[positions],
        "tf-idf + logistic regression": tfidf_pred[positions],
        f"{MODEL}, zero-shot": answered["pred"].astype(int).to_numpy(),
    }
    latency = {
        "majority constant": "—",
        "hand-written rule": f"{rule_us_per_doc / 1e3:.4f} ms",
        "tf-idf + logistic regression": f"{tfidf_us_batched / 1e3:.4f} ms",
        f"{MODEL}, zero-shot": f"{1e3 * answered['secs'].median():.1f} ms",
    }
    rows = []
    for name, p in preds.items():
        s = score(y_s, p)
        lo, hi = bootstrap_macro_f1(y_s, p)
        rows.append([name, s["accuracy"], s["macro-F1"], f"[{lo:.1f}, {hi:.1f}]", s["F1(neg)"], latency[name]])
    table = pd.DataFrame(rows, columns=["approach", "accuracy", "macro-F1", "95% CI",
                                        "F1(neg)", "latency per review"]).set_index("approach")
    print(f"{len(y_s)} reviews: {int((y_s == 0).sum())} negative, {int((y_s == 1).sum())} positive\n")
    print(table.round(2).to_string())
    print(f"\nthe LLM's median latency is {1e3 * answered['secs'].median() / (tfidf_us_batched / 1e3):,.0f}x "
          "the tf-idf model's batched latency per review")
    print(f"\n{MODEL} confusion matrix:")
    show_confusion(y_s, preds[f"{MODEL}, zero-shot"])

    # what the balanced sample hides: precision on complaints at the corpus's own mix
    prevalence = (y_test == 0).mean()
    measured_on_full = {"hand-written rule": rule_pred, "tf-idf + logistic regression": tfidf_pred}
    print(f"\ncomplaint precision at the full test set's {100 * prevalence:.2f}% complaint rate")
    print(f"{'':<30}{'recall(neg)':>12}{'false alarms':>14}{'projected':>11}{'measured, n=10,000':>20}")
    for name, p in list(preds.items())[1:]:
        recall = (p[y_s == 0] == 0).mean()
        false_alarm = (p[y_s == 1] == 0).mean()
        denom = recall * prevalence + false_alarm * (1 - prevalence)
        projected = f"{100 * recall * prevalence / denom:.1f}" if denom > 0 else "—"
        if name in measured_on_full:
            full_pred = measured_on_full[name]
            measured = f"{100 * ((full_pred == 0) & (y_test == 0)).sum() / (full_pred == 0).sum():.1f}"
        else:
            measured = "not run"
        print(f"{name:<30}{100 * recall:>11.1f}%{100 * false_alarm:>13.1f}%{projected:>11}{measured:>20}")

100 reviews: 50 negative, 50 positive

                              accuracy  macro-F1        95% CI  F1(neg) latency per review
approach                                                                                  
majority constant                 50.0     33.33  [28.6, 37.5]     0.00                  —
hand-written rule                 61.0     54.00  [43.4, 63.8]    36.07          0.0086 ms
tf-idf + logistic regression      86.0     85.91  [78.6, 92.8]    84.78          0.0168 ms
llama3.2:3b, zero-shot            89.0     89.00  [82.4, 95.0]    88.89           104.8 ms

the LLM's median latency is 6,252x the tf-idf model's batched latency per review

llama3.2:3b confusion matrix:
            predicted neg  predicted pos
actual neg             44              6
actual pos              5             45

complaint precision at the full test set's 6.32% complaint rate
                               recall(neg)  false alarms  projected  measured, n=10,000
hand-written rule         

**Reading it.** Check the parse count before any score: a score over answers that did not
parse is not a score. Then read the four rows against the 50% baseline of this sample,
and look at how wide the intervals are. With 100 reviews, rows whose intervals overlap
heavily have not been separated. Chapter 12 uses 2,000 documents and a *paired*
bootstrap for exactly this reason.

**What the balanced sample hides.** Half of these 100 reviews are complaints; in the
corpus the positive class is 14.8 times larger. A false alarm on a positive review costs
little on this sample and a great deal on real traffic. The last printout projects each
approach's complaint precision at the full test set's rate, from its recall and
false-alarm rate on the sample. For the rule and TF-IDF you can check the projection
against the precision measured on all 10,000 reviews. With 50 positive reviews the
false-alarm rate moves in steps of 2 points, so read the projection coarsely: a
false-alarm rate of zero projects to a perfect precision that 50 reviews cannot support,
and the measured column shows what 10,000 reviews say instead.

**Context, not a reproduction.** Chapter 12 scored `llama3.2:3b` with its own
English prompt at 69.21 macro-F1, 31.56% complaint precision and 0.124 s per document
(median) on a stratified 2,000-review subsample. Different prompt, different sample,
different class balance: the numbers above are not comparable to it.

The point the chapter's table makes is in the last column. The prompted model needed no
training labels, and it costs orders of magnitude more time per review than a model that
trained in about a second. The ratio printed above is the part that travels between
machines. That is the trade the chapter describes: **flexibility, paid for in cost,
latency and predictability.** [Chapter 12](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html#sec-classification-crossover)
measures where that trade flips.

## 4. Agreement is not reliability

Intuition check 3 asks: *two annotators agree on 93% of your labels — are the labels
trustworthy?* Not necessarily. When one label dominates, two labellers who never read a
thing still agree most of the time. **Cohen's kappa** subtracts the agreement they would
reach labelling independently at their own label rates:

$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

where $p_o$ is the observed agreement and $p_e$ is the agreement expected if both
annotators labelled independently, each at their own frequencies.

### 4a. Reproducing the book's numbers from its table

[Chapter 5](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html) compares the gold labels (annotator A) with a crude lexicon rule
(annotator B) on the 10,000 test reviews and prints the confusion table. The cell rebuilds
$p_o$, $p_e$ and kappa from those four counts alone, then checks the result against
scikit-learn's `cohen_kappa_score`. The book gives $p_o$ = 0.9342, $p_e$ = 0.9099 and
kappa = 0.2698, with A calling 6.32% of items negative and B 3.08%.

In [13]:
def agreement(table):
    """p_o, p_e and Cohen's kappa from a 2x2 table: rows annotator A, columns annotator B."""
    table = np.asarray(table, dtype=float)
    n = table.sum()
    p_o = np.trace(table) / n
    a_rates = table.sum(axis=1) / n          # how often A uses each label
    b_rates = table.sum(axis=0) / n          # how often B uses each label
    p_e = float(a_rates @ b_rates)
    return {"p_o": p_o, "p_e": p_e, "kappa": (p_o - p_e) / (1 - p_e),
            "A negative": a_rates[0], "B negative": b_rates[0]}


# chapter 5: A (gold, rows) vs B (lexicon rule, columns), negative first
ch05_table = [[141, 491],
              [167, 9201]]
ch05 = agreement(ch05_table)

vs_book("A calls negative, %", 100 * ch05["A negative"], 6.32, "{:.2f}")
vs_book("B calls negative, %", 100 * ch05["B negative"], 3.08, "{:.2f}")
vs_book("p_o", ch05["p_o"], 0.9342, "{:.4f}")
vs_book("p_e", ch05["p_e"], 0.9099, "{:.4f}")
vs_book("kappa", ch05["kappa"], 0.2698, "{:.4f}")

# the same kappa from label vectors expanded out of the table
a = np.repeat([0, 0, 1, 1], [141, 491, 167, 9201])
b = np.repeat([0, 1, 0, 1], [141, 491, 167, 9201])
print(f"\nscikit-learn cohen_kappa_score on the expanded table: {cohen_kappa_score(a, b):.4f}")
print(f"above the baseline: {100 * (ch05['p_o'] - ch05['p_e']):.2f} points, "
      f"out of {100 * (1 - ch05['p_e']):.2f} available")

A calls negative, %            this run      6.32   book      6.32   same as the book
B calls negative, %            this run      3.08   book      3.08   same as the book
p_o                            this run    0.9342   book    0.9342   same as the book
p_e                            this run    0.9099   book    0.9099   same as the book
kappa                          this run    0.2698   book    0.2698   same as the book

scikit-learn cohen_kappa_score on the expanded table: 0.2698
above the baseline: 2.43 points, out of 9.01 available


**Reading it.** Chapter 1 reads these numbers as a comparison, not a decomposition: 93.42
points of observed agreement against the 90.99 that two labellers with these rates would
reach labelling independently. That leaves 2.43 points above the baseline, out of only
9.01 available, and kappa is the share of that headroom taken: about 0.27.

$p_e$ is what an independence null predicts. It does not say which 90.99 points were
chance.

### 4b. Two doctors who guess

The chapter's analogy: 95% of the patients arriving for a screening are healthy. Two
doctors never look at a chart and each says "healthy" 95% of the time, independently.
They land on the same call $0.95^2 + 0.05^2 = 90.5\%$ of the time.

The cell simulates 100,000 patients for that case and for a few other guessing rates.
**What to look at:** raw agreement climbs as the guessing rate becomes more lopsided,
while kappa stays at about zero every time, because neither doctor is using any
information.

In [14]:
rng = np.random.default_rng(SEED)
n_patients = 100_000
rows = []
for rate in [0.50, 0.80, 0.95, 0.99]:
    doctor_1 = (rng.random(n_patients) < rate).astype(int)   # 1 = "healthy"
    doctor_2 = (rng.random(n_patients) < rate).astype(int)
    rows.append([rate, 100 * (rate**2 + (1 - rate)**2), 100 * (doctor_1 == doctor_2).mean(),
                 cohen_kappa_score(doctor_1, doctor_2)])
doctors = pd.DataFrame(rows, columns=["says healthy", "agreement, formula %",
                                      "agreement, simulated %", "kappa"])
print(doctors.round({"agreement, formula %": 2, "agreement, simulated %": 2, "kappa": 4}).to_string(index=False))

 says healthy  agreement, formula %  agreement, simulated %   kappa
         0.50                 50.00                   50.28  0.0055
         0.80                 68.00                   67.93 -0.0029
         0.95                 90.50                   90.53 -0.0007
         0.99                 98.02                   97.97 -0.0034


### 4c. Your own annotators

Now the same arithmetic on this notebook's own labels. Treat the gold column as annotator
A, and each approach from section 3 as annotator B, on all 10,000 test reviews.

The caveat chapter 5 attaches to its own version applies here too. None of these annotators is a
second human, so kappa here measures *how well each approach approximates the gold
labels*, not whether the labelling task is well defined. The arithmetic is identical
either way.

**What to look at:** order the three rows by raw agreement, then by kappa.

In [15]:
rows = []
for name, pred in [("majority constant", majority_pred),
                   ("hand-written rule", rule_pred),
                   ("tf-idf + logistic regression", tfidf_pred)]:
    tbl = confusion_matrix(y_test, pred, labels=[0, 1])
    ag = agreement(tbl)
    assert abs(ag["kappa"] - cohen_kappa_score(y_test, pred)) < 1e-12
    rows.append([name, 100 * ag["p_o"], 100 * ag["p_e"], ag["kappa"], 100 * ag["B negative"]])

print("A (gold, rows) vs B (hand-written rule, columns), n = 10,000")
print(pd.DataFrame(confusion_matrix(y_test, rule_pred, labels=[0, 1]),
                   index=["A: negative", "A: positive"],
                   columns=["B: negative", "B: positive"]).to_string())
print()
kappas = pd.DataFrame(rows, columns=["annotator B", "raw agreement %", "p_e %", "kappa",
                                     "B calls negative %"]).set_index("annotator B")
print(kappas.round({"raw agreement %": 2, "p_e %": 2, "kappa": 4, "B calls negative %": 2}).to_string())

A (gold, rows) vs B (hand-written rule, columns), n = 10,000
             B: negative  B: positive
A: negative          200          432
A: positive          303         9065

                              raw agreement %  p_e %   kappa  B calls negative %
annotator B                                                                     
majority constant                       93.68  93.68  0.0000                0.00
hand-written rule                       92.65  89.29  0.3140                5.03
tf-idf + logistic regression            92.57  84.29  0.5271               10.75


**Reading it.** Raw agreement with the gold labels *is* accuracy, so the ranking by raw
agreement puts the constant first: it agrees with gold most often, having read nothing.
Its kappa is exactly zero, because its $p_e$ equals its $p_o$. In this run kappa
reverses the whole order: the constant drops to last, and TF-IDF, the row with the
*lowest* raw agreement, comes first.

That is the chapter's general lesson arriving from a different direction: **a percentage
is meaningless without the baseline it should be read against.** Chapter 5 adds two
warnings worth carrying with it. Kappa is unstable when classes are this skewed, because a
handful of items moving between cells swings its small denominator a lot. And kappa inflates when two
annotators use the minority label at very different rates. Report the confusion table
beside any single coefficient.

## 5. Exercises

These extend [chapter 1's own exercises](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/01-why-nlp-now.html) and point to the chapters that go
deeper.

1. **Your own baseline.** Chapter 1's first exercise asks you to pick a document type from
   your work and say what "success" would mean as a number. If you have any labelled
   examples of it, compute the majority-class rate before anything else, then write down
   the accuracy a model would need before it has beaten a constant. Chapter 1's second
   interview question gives the order of work; [chapter 4](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/04-bake-off.html) is a whole chapter
   of what the cheap model scores.

2. **The tail, by hand.** Add `"sorun"` ("problem") to `NEG` in section 3a and re-run the
   section. Look at what happens to reviews containing `sorunsuz` ("problem-free"). Repair
   it, then count how many rules you have written and how many complaints are still
   missed. That is the rule-based era's failure in miniature.
   [Chapter 2](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/02-text-as-data.html#sec-text-as-data-zipf) explains why the vocabulary never
   closes, and chapter 4 counts 895 distinct word forms sharing the prefix `kulla` in
   this corpus's training split.

3. **How many labels does the classical column need?** The chapter's table says
   "thousands". Refit section 3b on stratified subsets of `train` with 200, 1,000 and 5,000
   reviews, and score each on the full test set. Compare your curve with
   [chapter 12's baseline curve](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/12-classification.html#sec-classification-baseline), then with the
   row of the table you are testing.

4. **Change one thing in the prompt.** Rewrite `PROMPT` in section 3c in Turkish and re-run
   it. Chapter 12 measured an English instruction as worth +16.90 macro-F1 on this model;
   does your run point the same way, and can 100 reviews tell? Then raise the sample to 300
   and watch the intervals narrow. [Chapter 17](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/17-prompting.html) covers prompting and
   [chapter 18](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/18-decoding-structured-output.html) covers constraining the output, which the chapter's
   table says prompted output needs.

5. **Same annotators, different sample.** Chapter 5's fifth exercise: compute kappa between
   the hand-written rule and gold on a class-balanced subsample (all 632 negatives plus 632
   positives drawn with a fixed seed) and compare it with section 4c. Explain the change
   using [chapter 5](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/05-data-annotation.html)'s prevalence and bias paradoxes, and say which number you
   would put in a report.

6. **A claim in the wild.** Chapter 1's fourth exercise: find a public claim about AI
   capability from the last month and locate the evaluation underneath it. Ask the three
   questions from the chapter's fifth interview question: what is the baseline, how big is
   the test set, and who wrote the labels? [Chapter 22](https://bbardakk.github.io/uygulamali-dogal-dil-isleme/en/chapters/22-evaluating-llm-systems.html) is where evaluation
   gets its full treatment.

---

*Code in this notebook is MIT-licensed; the prose is CC BY 4.0. The Turkish product
reviews dataset is CC BY-SA 4.0 (Fatih Barmanbay, `fthbrmnby/turkish_product_reviews` on
Hugging Face).*